# Dataproc Serverless & Apache Iceberg — Demo

Companion notebook to the **Dataproc Serverless & Apache Iceberg Handbook**.

Dataproc Serverless batches run on remote, ephemeral compute — not inside this notebook's kernel — so this notebook's job is to **generate the PySpark script, submit it as a batch, monitor it to completion, and then query the resulting Iceberg table from BigQuery**, all from one place.

**Naming note (July 2026):** 'Dataproc Serverless' is now marketed together with cluster-based Dataproc as the **Managed Service for Apache Spark**, and BigLake metastore is now the **Lakehouse runtime catalog** (BigLake for Apache Iceberg). APIs, IAM roles, and `gcloud dataproc` / `gcloud biglake` commands are unchanged.

Run the cells top to bottom. Update the variables in the **Configuration** cell first.


## 0. Setup: install & authenticate


In [ ]:
!pip install --quiet google-cloud-dataproc google-cloud-bigquery google-cloud-storage


In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab.')
except ImportError:
    print('Not running in Colab -- make sure you have run `gcloud auth application-default login` locally.')


## 1. Configuration — edit before running


In [ ]:
PROJECT_ID = "project-2df9d2ad-b6f5-4ed5-997"
REGION = "us-central1"
BUCKET = f"{PROJECT_ID}-iceberg-demo"
WAREHOUSE = f"gs://{BUCKET}/warehouse"
CATALOG_NAME = "demo_catalog"
NAMESPACE = "sales_ns"
TABLE_NAME = "orders"

# Iceberg / connector jar versions -- keep these pinned deliberately (see handbook best practices)
ICEBERG_VERSION = "1.6.1"
CATALOG_JAR = f"gs://spark-lib/bigquery/iceberg-bigquery-catalog-{ICEBERG_VERSION}-1.0.2.jar"
SPARK_ICEBERG_PACKAGE = f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION}"

print(f"Project: {PROJECT_ID} | Region: {REGION} | Warehouse: {WAREHOUSE}")


## 2. Create the Cloud Storage warehouse bucket


In [ ]:
from google.cloud import storage

storage_client = storage.Client(project=PROJECT_ID)

try:
    bucket = storage_client.create_bucket(BUCKET, location=REGION)
    print(f"Bucket created: {bucket.name}")
except Exception as e:
    print(f"Bucket may already exist: {e}")


## 3. Grant the batch job's service account the roles it needs

Uses the Compute Engine default service account, which Dataproc Serverless batches use unless you specify a custom one.


In [ ]:
import google.auth
import google.auth.transport.requests
import requests

creds, _ = google.auth.default()
creds.refresh(google.auth.transport.requests.Request())
headers = {
    "Authorization": f"Bearer {creds.token}",
    "Content-Type": "application/json",
}

project_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
project_info = requests.get(project_url, headers=headers).json()
project_number = project_info["projectNumber"]
service_account = f"{project_number}[email protected]"
print(f"Service account: {service_account}")

get_policy = requests.post(f"{project_url}:getIamPolicy", headers=headers, json={}).json()
bindings = get_policy.get("bindings", [])

for role in ["roles/bigquery.dataEditor", "roles/storage.objectAdmin"]:
    binding = next((b for b in bindings if b["role"] == role), None)
    if binding is None:
        binding = {"role": role, "members": []}
        bindings.append(binding)
    member = f"serviceAccount:{service_account}"
    if member not in binding["members"]:
        binding["members"].append(member)

get_policy["bindings"] = bindings
set_resp = requests.post(f"{project_url}:setIamPolicy", headers=headers, json={"policy": get_policy})
print(set_resp.status_code, "-- IAM roles granted (or already present).")


## 4. Generate the PySpark script


In [ ]:
pyspark_script = f'''
from pyspark.sql import SparkSession

spark = SparkSession.builder \\
    .appName("CreateIcebergTable") \\
    .config("spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \\
    .config("spark.sql.catalog.{CATALOG_NAME}.catalog-impl",
            "org.apache.iceberg.gcp.bigquery.BigQueryMetastoreCatalog") \\
    .config("spark.sql.catalog.{CATALOG_NAME}.gcp_project", "{PROJECT_ID}") \\
    .config("spark.sql.catalog.{CATALOG_NAME}.gcp_location", "{REGION}") \\
    .config("spark.sql.catalog.{CATALOG_NAME}.warehouse", "{WAREHOUSE}") \\
    .getOrCreate()

spark.sql("USE {CATALOG_NAME};")
spark.sql("CREATE NAMESPACE IF NOT EXISTS {NAMESPACE};")
spark.sql("USE {NAMESPACE};")
spark.sql("""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
      order_id INT,
      customer STRING,
      amount DOUBLE,
      status STRING
    ) USING ICEBERG
""")

spark.sql("""
    INSERT INTO {TABLE_NAME} VALUES
      (1, 'Riya Sharma', 4500.0, 'PLACED'),
      (2, 'Arjun Verma', 1200.0, 'PLACED'),
      (3, 'Meera Nair', 8900.0, 'SHIPPED')
""")

spark.sql("SELECT * FROM {TABLE_NAME}").show()
'''

with open("create_iceberg_table.py", "w") as f:
    f.write(pyspark_script)

print("Script written to create_iceberg_table.py")
print(pyspark_script)

In [ ]:
from google.cloud import storage as gcs

gcs_client = gcs.Client(project=PROJECT_ID)
bucket_obj = gcs_client.bucket(BUCKET)
blob = bucket_obj.blob("scripts/create_iceberg_table.py")
blob.upload_from_filename("create_iceberg_table.py")
SCRIPT_URI = f"gs://{BUCKET}/scripts/create_iceberg_table.py"
print(f"Uploaded script to {SCRIPT_URI}")


## 5. Submit the batch job


In [ ]:
from google.cloud import dataproc_v1

batch_client = dataproc_v1.BatchControllerClient(
    client_options={"api_endpoint": f"{REGION}-dataproc.googleapis.com:443"}
)

batch = dataproc_v1.Batch()
batch.pyspark_batch.main_python_file_uri = SCRIPT_URI
batch.pyspark_batch.jar_file_uris = [CATALOG_JAR]
batch.runtime_config.version = "2.2"
batch.runtime_config.properties = {
    "spark.jars.packages": SPARK_ICEBERG_PACKAGE,
}

parent = f"projects/{PROJECT_ID}/locations/{REGION}"
operation = batch_client.create_batch(parent=parent, batch=batch)
print("Batch submitted. Waiting for it to complete (this can take a few minutes)...")

result = operation.result()  # blocks until the batch finishes
print("Batch finished with state:", result.state)


## 6. View the driver output logs

Confirms the three inserted rows were written successfully.


In [ ]:
print("Batch resource name:", result.name)
print("Check full logs in the console: https://console.cloud.google.com/dataproc/batches")
print("Or via CLI:")
batch_id = result.name.split("/")[-1]
print(f"gcloud dataproc batches describe {batch_id} --region={REGION}")


## 7. Query the same Iceberg table from BigQuery

Because Spark wrote table metadata into BigLake metastore (Lakehouse runtime catalog), the table is immediately queryable from BigQuery SQL — no export or load job needed.


In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)

query = f"""
# SELECT * FROM `{PROJECT_ID}.{CATALOG_NAME}.{NAMESPACE}.{TABLE_NAME}`
SELECT * FROM `project-2df9d2ad-b6f5-4ed5-997.sales_ns.orders`
"""

try:
    for row in bq_client.query(query).result():
        print(dict(row))
except Exception as e:
    print("If this fails, check BigQuery Studio's Explorer panel for the auto-discovered")
    print("catalog/dataset path -- the exact reference syntax depends on how the BigLake")
    print("catalog is registered as a BigQuery connection.")
    print("Error:", e)


## 8. Cleanup


In [ ]:
CONFIRM_CLEANUP = True  # flip to True to actually delete resources

if CONFIRM_CLEANUP:
    blobs = list(bucket_obj.list_blobs())
    for b in blobs:
        b.delete()
    bucket_obj.delete()
    print("Warehouse bucket and contents deleted.")
else:
    print("Skipped. Set CONFIRM_CLEANUP = True and re-run this cell to clean up.")
